# 01 Data Validation

This notebook validates the CSV files in `data/` for the property management analytics project.

Initial inspection found 12 CSV files with 69,125 total rows. The operational tables have stable primary keys, clean foreign-key joins to the property base, and expected 2025 monthly coverage. The raw NYC DOF source has 3 duplicate rows; the curated property base does not.

Run the cells from top to bottom after refreshing the data folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.options.display.float_format = "{:,.2f}".format

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR

PosixPath('<project-root>/data')

In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
tables = {path.stem: pd.read_csv(path, low_memory=False) for path in csv_files}

print(f"Loaded {len(csv_files)} CSV files from {DATA_DIR}")
for path in csv_files:
    print(f"- {path.name}")

Loaded 12 CSV files from <project-root>/data
- DOF_Condominium_Comparable_Rental_Income_in_NYC.csv
- accounts_receivable.csv
- ai_monthly_summary_inputs.csv
- compliance_reviews.csv
- leasing_pipeline.csv
- monthly_financials.csv
- property_managers.csv
- source_real_estate_base_properties.csv
- tenants.csv
- vendor_invoices.csv
- vendors.csv
- work_orders.csv


## File Inventory

This profile checks basic file shape: row counts, column counts, duplicate rows, and missing-cell volume.

In [3]:
file_profile = []
for name, df in tables.items():
    file_profile.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_columns": int((df.isna().sum() > 0).sum()),
    })

file_profile = pd.DataFrame(file_profile).sort_values("table").reset_index(drop=True)
display(file_profile)

,table,rows,columns,duplicate_rows,missing_cells,missing_columns
0,DOF_Condominium_Comparable_Rental_Income_in_NYC,22073,12,3,76,3
1,accounts_receivable,25740,10,0,0,0
2,ai_monthly_summary_inputs,12,21,0,0,0
3,compliance_reviews,2200,11,0,0,0
4,leasing_pipeline,179,11,0,0,0
5,monthly_financials,2100,16,0,0,0
6,property_managers,10,5,0,0,0
7,source_real_estate_base_properties,175,20,0,0,0
8,tenants,2145,12,0,0,0
9,vendor_invoices,10581,17,0,685,2


In [4]:
missing_profile = []
for name, df in tables.items():
    null_counts = df.isna().sum()
    for column, null_count in null_counts[null_counts > 0].items():
        missing_profile.append({
            "table": name,
            "column": column,
            "missing_rows": int(null_count),
            "missing_pct": null_count / len(df),
        })

missing_profile = pd.DataFrame(missing_profile)
if missing_profile.empty:
    print("No missing values found.")
else:
    display(missing_profile.sort_values(["table", "missing_rows"], ascending=[True, False]))

,table,column,missing_rows,missing_pct
0,DOF_Condominium_Comparable_Rental_Income_in_NYC,Year Built,73,0.00
2,DOF_Condominium_Comparable_Rental_Income_in_NYC,Full Market Value,2,0.00
1,DOF_Condominium_Comparable_Rental_Income_in_NYC,Estimated Expense,1,0.00
4,vendor_invoices,gl_code,365,0.03
3,vendor_invoices,paid_date,320,0.03
5,work_orders,tenant_id,1362,0.35
6,work_orders,closed_date,848,0.22
7,work_orders,days_to_close,848,0.22


## Validation Setup

The expected schemas come from the data dictionary and the current inspected CSV headers.

In [5]:
EXPECTED = {
    "DOF_Condominium_Comparable_Rental_Income_in_NYC": {
        "rows": 22073,
        "primary_key": None,
        "columns": [
            "Boro-Block-Lot", "Address", "Neighborhood", "Building Classification",
            "Total Units", "Year Built", "Gross SqFt", "Estimated Gross Income",
            "Estimated Expense", "Net Operating Income", "Full Market Value", "Report Year",
        ],
    },
    "source_real_estate_base_properties": {
        "rows": 175,
        "primary_key": "property_id",
        "columns": [
            "property_id", "source_boro_block_lot", "address", "borough", "neighborhood",
            "building_classification", "asset_type", "total_units", "year_built", "gross_sqft",
            "base_estimated_gross_income", "base_estimated_expense", "base_net_operating_income",
            "base_full_market_value", "source_report_year", "property_manager_id", "client_account_id",
            "asset_status", "source_dataset", "source_is_real",
        ],
    },
    "property_managers": {
        "rows": 10,
        "primary_key": "property_manager_id",
        "columns": ["property_manager_id", "property_manager_name", "market_region", "assignment_start_date", "synthetic_record"],
    },
    "tenants": {
        "rows": 2145,
        "primary_key": "tenant_id",
        "columns": [
            "tenant_id", "property_id", "tenant_name", "tenant_category", "occupied_sqft", "unit_count",
            "lease_start_date", "lease_end_date", "monthly_rent", "security_deposit", "tenant_status", "synthetic_record",
        ],
    },
    "vendors": {
        "rows": 17,
        "primary_key": "vendor_id",
        "columns": ["vendor_id", "vendor_name", "default_invoice_category", "synthetic_record"],
    },
    "monthly_financials": {
        "rows": 2100,
        "primary_key": "financial_id",
        "columns": [
            "financial_id", "property_id", "month", "budget_income", "actual_income", "income_variance",
            "income_variance_pct", "budget_expenses", "actual_expenses", "expense_variance", "expense_variance_pct",
            "net_operating_income", "ebitda", "occupancy_rate", "management_fee", "synthetic_record",
        ],
    },
    "accounts_receivable": {
        "rows": 25740,
        "primary_key": "ar_id",
        "columns": [
            "ar_id", "property_id", "tenant_id", "month", "billed_rent", "amount_paid", "outstanding_balance",
            "aging_bucket", "collection_priority", "synthetic_record",
        ],
    },
    "vendor_invoices": {
        "rows": 10581,
        "primary_key": "invoice_id",
        "columns": [
            "invoice_id", "property_id", "vendor_id", "vendor_name", "invoice_number", "invoice_category",
            "invoice_date", "due_date", "paid_date", "invoice_amount", "approval_status", "gl_code",
            "late_payment_flag", "potential_duplicate_invoice_flag", "missing_gl_code_flag", "compliance_flag", "synthetic_record",
        ],
    },
    "work_orders": {
        "rows": 3893,
        "primary_key": "work_order_id",
        "columns": [
            "work_order_id", "property_id", "tenant_id", "opened_date", "closed_date", "work_order_category",
            "priority", "status", "sla_days", "days_to_close", "overdue_flag", "estimated_cost", "vendor_id",
            "vendor_name", "synthetic_record",
        ],
    },
    "compliance_reviews": {
        "rows": 2200,
        "primary_key": "review_id",
        "columns": [
            "review_id", "property_id", "month", "review_type", "issue_category", "severity",
            "finding_description", "recommendation", "status", "owner", "synthetic_record",
        ],
    },
    "leasing_pipeline": {
        "rows": 179,
        "primary_key": "opportunity_id",
        "columns": [
            "opportunity_id", "property_id", "prospect_name", "stage", "probability", "expected_monthly_rent",
            "weighted_monthly_rent", "expected_lease_start", "space_sqft", "source", "synthetic_record",
        ],
    },
    "ai_monthly_summary_inputs": {
        "rows": 12,
        "primary_key": None,
        "columns": [
            "month", "portfolio_budget_income", "portfolio_actual_income", "portfolio_budget_expenses",
            "portfolio_actual_expenses", "portfolio_noi", "portfolio_ebitda", "avg_occupancy",
            "total_ar_outstanding", "critical_ar_count", "high_ar_count", "invoice_count", "invoice_amount",
            "invoice_exception_count", "late_invoice_count", "work_order_count", "overdue_work_order_count",
            "avg_estimated_cost", "income_variance", "expense_variance", "ai_prompt_ready_note",
        ],
    },
}

ALLOWED_NULL_COLUMNS = {
    "DOF_Condominium_Comparable_Rental_Income_in_NYC": {"Year Built", "Full Market Value", "Estimated Expense"},
    "vendor_invoices": {"gl_code", "paid_date"},
    "work_orders": {"tenant_id", "closed_date", "days_to_close"},
}

DATE_COLUMNS = {
    "DOF_Condominium_Comparable_Rental_Income_in_NYC": ["Report Year"],
    "source_real_estate_base_properties": ["year_built", "source_report_year"],
    "property_managers": ["assignment_start_date"],
    "tenants": ["lease_start_date", "lease_end_date"],
    "monthly_financials": ["month"],
    "accounts_receivable": ["month"],
    "vendor_invoices": ["invoice_date", "due_date", "paid_date"],
    "work_orders": ["opened_date", "closed_date"],
    "compliance_reviews": ["month"],
    "leasing_pipeline": ["expected_lease_start"],
    "ai_monthly_summary_inputs": ["month"],
}

EXPECTED_MONTHS = [f"2025-{month:02d}" for month in range(1, 13)]

In [6]:
validation_results = []

def add_check(area, check, status, detail=""):
    validation_results.append({
        "area": area,
        "check": check,
        "status": status,
        "detail": detail,
    })

def missing_fk(child_df, child_col, parent_df, parent_col):
    child_values = set(child_df[child_col].dropna().astype(str))
    parent_values = set(parent_df[parent_col].dropna().astype(str))
    return sorted(child_values - parent_values)

def close_enough(actual, expected, tolerance=0.02):
    return (actual.sub(expected).abs() <= tolerance) | (actual.isna() & expected.isna())

def month_strings(series):
    return sorted(pd.to_datetime(series).dt.to_period("M").astype(str).unique().tolist())

## Schema, Duplicates, Nulls, and Primary Keys

In [7]:
expected_tables = set(EXPECTED)
actual_tables = set(tables)

missing_tables = sorted(expected_tables - actual_tables)
unexpected_tables = sorted(actual_tables - expected_tables)

add_check("inventory", "expected CSV files present", "PASS" if not missing_tables else "FAIL", f"missing={missing_tables}")
add_check("inventory", "no unexpected CSV files", "PASS" if not unexpected_tables else "WARN", f"unexpected={unexpected_tables}")

for table_name, spec in EXPECTED.items():
    if table_name not in tables:
        continue

    df = tables[table_name]
    expected_columns = spec["columns"]
    actual_columns = list(df.columns)

    add_check(
        table_name,
        "row count",
        "PASS" if len(df) == spec["rows"] else "WARN",
        f"actual={len(df):,}; expected={spec['rows']:,}",
    )
    add_check(
        table_name,
        "column order and names",
        "PASS" if actual_columns == expected_columns else "FAIL",
        f"actual_columns={actual_columns}",
    )

    duplicate_rows = int(df.duplicated().sum())
    duplicate_status = "PASS" if duplicate_rows == 0 else ("WARN" if table_name.startswith("DOF_") else "FAIL")
    add_check(table_name, "duplicate full rows", duplicate_status, f"duplicate_rows={duplicate_rows:,}")

    allowed_nulls = ALLOWED_NULL_COLUMNS.get(table_name, set())
    null_counts = df.isna().sum()
    unexpected_nulls = {col: int(count) for col, count in null_counts.items() if count > 0 and col not in allowed_nulls}
    expected_nulls = {col: int(count) for col, count in null_counts.items() if count > 0 and col in allowed_nulls}
    add_check(table_name, "unexpected null values", "PASS" if not unexpected_nulls else "FAIL", str(unexpected_nulls))
    if expected_nulls:
        add_check(table_name, "documented nullable fields", "WARN", str(expected_nulls))

    primary_key = spec.get("primary_key")
    if primary_key:
        duplicate_keys = int(df[primary_key].duplicated().sum())
        missing_keys = int(df[primary_key].isna().sum())
        add_check(
            table_name,
            f"primary key {primary_key}",
            "PASS" if duplicate_keys == 0 and missing_keys == 0 else "FAIL",
            f"duplicate_keys={duplicate_keys:,}; missing_keys={missing_keys:,}",
        )

## Relationships and Date Coverage

In [8]:
props = tables["source_real_estate_base_properties"]
property_managers = tables["property_managers"]
tenants = tables["tenants"]
vendors = tables["vendors"]
monthly = tables["monthly_financials"]
ar = tables["accounts_receivable"]
invoices = tables["vendor_invoices"]
work_orders = tables["work_orders"]
reviews = tables["compliance_reviews"]
pipeline = tables["leasing_pipeline"]
ai_summary = tables["ai_monthly_summary_inputs"]

fk_checks = [
    ("source_real_estate_base_properties", props, "property_manager_id", property_managers, "property_manager_id"),
    ("tenants", tenants, "property_id", props, "property_id"),
    ("monthly_financials", monthly, "property_id", props, "property_id"),
    ("accounts_receivable", ar, "property_id", props, "property_id"),
    ("accounts_receivable", ar, "tenant_id", tenants, "tenant_id"),
    ("vendor_invoices", invoices, "property_id", props, "property_id"),
    ("vendor_invoices", invoices, "vendor_id", vendors, "vendor_id"),
    ("work_orders", work_orders, "property_id", props, "property_id"),
    ("work_orders", work_orders.dropna(subset=["tenant_id"]), "tenant_id", tenants, "tenant_id"),
    ("work_orders", work_orders, "vendor_id", vendors, "vendor_id"),
    ("compliance_reviews", reviews, "property_id", props, "property_id"),
    ("leasing_pipeline", pipeline, "property_id", props, "property_id"),
]

for table_name, child_df, child_col, parent_df, parent_col in fk_checks:
    missing = missing_fk(child_df, child_col, parent_df, parent_col)
    add_check(
        table_name,
        f"foreign key {child_col} -> {parent_col}",
        "PASS" if not missing else "FAIL",
        f"missing_count={len(missing):,}; sample={missing[:10]}",
    )

for table_name, date_columns in DATE_COLUMNS.items():
    df = tables[table_name]
    for column in date_columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        source_non_null = df[column].notna()
        parse_failures = int((source_non_null & parsed.isna()).sum())
        add_check(
            table_name,
            f"date parse: {column}",
            "PASS" if parse_failures == 0 else "FAIL",
            f"parse_failures={parse_failures:,}",
        )

for table_name, df in {
    "monthly_financials": monthly,
    "accounts_receivable": ar,
    "compliance_reviews": reviews,
    "ai_monthly_summary_inputs": ai_summary,
}.items():
    months = month_strings(df["month"])
    add_check(table_name, "2025 month coverage", "PASS" if months == EXPECTED_MONTHS else "FAIL", str(months))

## Business Rule Checks

In [9]:
calculation_checks = [
    ("monthly_financials", "income_variance = actual_income - budget_income", monthly["income_variance"], monthly["actual_income"] - monthly["budget_income"], 0.02),
    ("monthly_financials", "expense_variance = actual_expenses - budget_expenses", monthly["expense_variance"], monthly["actual_expenses"] - monthly["budget_expenses"], 0.02),
    ("monthly_financials", "net_operating_income = actual_income - actual_expenses", monthly["net_operating_income"], monthly["actual_income"] - monthly["actual_expenses"], 0.02),
    ("monthly_financials", "ebitda = net_operating_income - management_fee", monthly["ebitda"], monthly["net_operating_income"] - monthly["management_fee"], 0.02),
    ("accounts_receivable", "outstanding_balance = billed_rent - amount_paid", ar["outstanding_balance"], ar["billed_rent"] - ar["amount_paid"], 0.02),
    ("leasing_pipeline", "weighted_monthly_rent = expected_monthly_rent * probability", pipeline["weighted_monthly_rent"], pipeline["expected_monthly_rent"] * pipeline["probability"], 0.02),
]

for table_name, check_name, actual, expected, tolerance in calculation_checks:
    failures = int((~close_enough(actual, expected, tolerance)).sum())
    add_check(table_name, check_name, "PASS" if failures == 0 else "FAIL", f"failures={failures:,}")

occupancy_failures = int(((monthly["occupancy_rate"] < 0) | (monthly["occupancy_rate"] > 1)).sum())
add_check("monthly_financials", "occupancy_rate between 0 and 1", "PASS" if occupancy_failures == 0 else "FAIL", f"failures={occupancy_failures:,}")

probability_failures = int(((pipeline["probability"] < 0) | (pipeline["probability"] > 1)).sum())
add_check("leasing_pipeline", "probability between 0 and 1", "PASS" if probability_failures == 0 else "FAIL", f"failures={probability_failures:,}")

lease_start = pd.to_datetime(tenants["lease_start_date"])
lease_end = pd.to_datetime(tenants["lease_end_date"])
lease_failures = int((lease_end < lease_start).sum())
add_check("tenants", "lease_end_date on or after lease_start_date", "PASS" if lease_failures == 0 else "FAIL", f"failures={lease_failures:,}")

missing_gl_flag_failures = int((invoices["missing_gl_code_flag"].astype(bool) != invoices["gl_code"].isna()).sum())
add_check("vendor_invoices", "missing_gl_code_flag matches null gl_code", "PASS" if missing_gl_flag_failures == 0 else "FAIL", f"failures={missing_gl_flag_failures:,}")

paid_invoices = invoices.dropna(subset=["paid_date"]).copy()
late_expected = pd.to_datetime(paid_invoices["paid_date"]) > pd.to_datetime(paid_invoices["due_date"])
late_flag_failures = int((paid_invoices["late_payment_flag"].astype(bool).to_numpy() != late_expected.to_numpy()).sum())
add_check("vendor_invoices", "late_payment_flag matches paid_date > due_date", "PASS" if late_flag_failures == 0 else "FAIL", f"failures={late_flag_failures:,}")

exact_invoice_duplicates = invoices.duplicated(subset=["vendor_id", "invoice_number", "invoice_amount"], keep=False)
duplicate_flag_mismatch = int((invoices["potential_duplicate_invoice_flag"].astype(bool) != exact_invoice_duplicates).sum())
add_check(
    "vendor_invoices",
    "potential_duplicate_invoice_flag vs exact duplicate invoice triplets",
    "PASS" if duplicate_flag_mismatch == 0 else "WARN",
    f"flag_mismatches={duplicate_flag_mismatch:,}; exact_duplicate_rows={int(exact_invoice_duplicates.sum()):,}",
)

closed_work_orders = work_orders[work_orders["status"].eq("Closed")].copy()
open_work_orders = work_orders[~work_orders["status"].eq("Closed")].copy()
closed_missing_date = int(closed_work_orders["closed_date"].isna().sum())
open_with_closed_date = int(open_work_orders["closed_date"].notna().sum())
add_check(
    "work_orders",
    "closed_date aligns to work order status",
    "PASS" if closed_missing_date == 0 and open_with_closed_date == 0 else "FAIL",
    f"closed_missing_date={closed_missing_date:,}; open_with_closed_date={open_with_closed_date:,}",
)

overdue_expected = closed_work_orders["days_to_close"] > closed_work_orders["sla_days"]
overdue_failures = int((closed_work_orders["overdue_flag"].astype(bool).to_numpy() != overdue_expected.to_numpy()).sum())
add_check("work_orders", "overdue_flag matches days_to_close > sla_days", "PASS" if overdue_failures == 0 else "FAIL", f"failures={overdue_failures:,}")

## AI Monthly Summary Reconciliation

The AI summary input table should reconcile to monthly financials, AR, invoices, and work orders. Small rounding differences are allowed.

In [10]:
financials_by_month = monthly.groupby("month", as_index=False).agg(
    portfolio_budget_income=("budget_income", "sum"),
    portfolio_actual_income=("actual_income", "sum"),
    portfolio_budget_expenses=("budget_expenses", "sum"),
    portfolio_actual_expenses=("actual_expenses", "sum"),
    portfolio_noi=("net_operating_income", "sum"),
    portfolio_ebitda=("ebitda", "sum"),
    avg_occupancy=("occupancy_rate", "mean"),
    income_variance=("income_variance", "sum"),
    expense_variance=("expense_variance", "sum"),
)

ar_by_month = ar.groupby("month", as_index=False).agg(
    total_ar_outstanding=("outstanding_balance", "sum"),
    critical_ar_count=("collection_priority", lambda s: int((s == "Critical").sum())),
    high_ar_count=("collection_priority", lambda s: int((s == "High").sum())),
)

invoice_month = invoices.assign(month=pd.to_datetime(invoices["invoice_date"]).dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d"))
invoices_by_month = invoice_month.groupby("month", as_index=False).agg(
    invoice_count=("invoice_id", "count"),
    invoice_amount=("invoice_amount", "sum"),
    invoice_exception_count=("compliance_flag", lambda s: int(s.astype(bool).sum())),
    late_invoice_count=("late_payment_flag", lambda s: int(s.astype(bool).sum())),
)

opened_month = work_orders.assign(month=pd.to_datetime(work_orders["opened_date"]).dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d"))
work_orders_by_month = opened_month.groupby("month", as_index=False).agg(
    work_order_count=("work_order_id", "count"),
    overdue_work_order_count=("overdue_flag", lambda s: int(s.astype(bool).sum())),
    avg_estimated_cost=("estimated_cost", "mean"),
)

reconciled_summary = (
    financials_by_month
    .merge(ar_by_month, on="month", how="inner")
    .merge(invoices_by_month, on="month", how="inner")
    .merge(work_orders_by_month, on="month", how="inner")
)

summary_compare = ai_summary.merge(reconciled_summary, on="month", suffixes=("_csv", "_calc"))
tolerances = {
    "avg_occupancy": 0.005,
    "avg_estimated_cost": 0.01,
}
default_tolerance = 0.25

reconciliation_rows = []
for column in [col for col in ai_summary.columns if col not in {"month", "ai_prompt_ready_note"}]:
    if column not in reconciled_summary.columns:
        continue
    diff = (summary_compare[f"{column}_csv"] - summary_compare[f"{column}_calc"]).abs()
    tolerance = tolerances.get(column, default_tolerance)
    failures = int((diff > tolerance).sum())
    reconciliation_rows.append({
        "metric": column,
        "max_abs_diff": float(diff.max()),
        "tolerance": tolerance,
        "failures": failures,
    })
    add_check(
        "ai_monthly_summary_inputs",
        f"reconcile {column}",
        "PASS" if failures == 0 else "FAIL",
        f"max_abs_diff={float(diff.max()):,.4f}; tolerance={tolerance}",
    )

reconciliation_report = pd.DataFrame(reconciliation_rows)
display(reconciliation_report)

,metric,max_abs_diff,tolerance,failures
0,portfolio_budget_income,0.00,0.25,0
1,portfolio_actual_income,0.00,0.25,0
2,portfolio_budget_expenses,0.00,0.25,0
3,portfolio_actual_expenses,0.00,0.25,0
4,portfolio_noi,0.00,0.25,0
5,portfolio_ebitda,0.00,0.25,0
6,avg_occupancy,0.00,0.01,0
7,total_ar_outstanding,0.00,0.25,0
8,critical_ar_count,0.00,0.25,0
9,high_ar_count,0.00,0.25,0


## Validation Results

In [11]:
results = pd.DataFrame(validation_results)
status_order = pd.CategoricalDtype(["FAIL", "WARN", "PASS"], ordered=True)
results["status"] = results["status"].astype(status_order)

summary = (
    results.groupby(["area", "status"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(summary)

exceptions = results[results["status"].isin(["FAIL", "WARN"])].sort_values(["status", "area", "check"])
if exceptions.empty:
    print("All validation checks passed.")
else:
    display(exceptions)

status,area,FAIL,WARN,PASS
0,DOF_Condominium_Comparable_Rental_Income_in_NYC,0,2,4
1,accounts_receivable,0,0,10
2,ai_monthly_summary_inputs,0,0,25
3,compliance_reviews,0,0,8
4,inventory,0,0,2
5,leasing_pipeline,0,0,9
6,monthly_financials,0,0,13
7,property_managers,0,0,6
8,source_real_estate_base_properties,0,0,8
9,tenants,0,0,9


,area,check,status,detail
6,DOF_Condominium_Comparable_Rental_Income_in_NYC,documented nullable fields,WARN,"{'Year Built': 73, 'Estimated Expense': 1, 'Fu..."
4,DOF_Condominium_Comparable_Rental_Income_in_NYC,duplicate full rows,WARN,duplicate_rows=3
41,vendor_invoices,documented nullable fields,WARN,"{'paid_date': 320, 'gl_code': 365}"
106,vendor_invoices,potential_duplicate_invoice_flag vs exact dupl...,WARN,flag_mismatches=194; exact_duplicate_rows=0
47,work_orders,documented nullable fields,WARN,"{'tenant_id': 1362, 'closed_date': 848, 'days_..."


## Current Inspection Notes

- All operational primary keys were unique during the initial inspection.
- Core foreign keys resolved cleanly: `property_id`, `tenant_id`, `vendor_id`, and `property_manager_id` had no orphaned populated values.
- Monthly operational tables covered January through December 2025.
- Expected nullable fields appeared in the raw DOF source, unpaid invoice dates or GL codes, and open/non-tenant work order fields.
- The raw DOF source contained 3 duplicate full rows. The curated `source_real_estate_base_properties.csv` table had no duplicate rows.
- `potential_duplicate_invoice_flag` marks 194 invoice rows for review, but none are exact duplicates by `vendor_id`, `invoice_number`, and `invoice_amount`. Treat this as a potential-duplicate review flag unless the project defines stricter duplicate criteria.